[Git Repository](https://github.com/ljosnorth/ind320ljosnorth/blob/main/streamlit_app.py)\
[Streamlit website](https://github.com/LjosNorth/IND320LjosNorth/tree/main)

### AI usage
I in my coding work utilize AI in four primary ways.
1. Inline suggestions. these help code faster, though ofc often they are also completely wrong so its not a brain off usage thing and one must be aware of what one is accepting.
2. alt to reading docs. If im wondering what the method .parent does from Pathlib one would traditionally look up the documentation and read it. I might still do that but often i'll also just ask a LLM.
3. how-to's. I will often have some idea of i want to do this thing but im not sure how to do it, say how to do subplots with plotly (my experience is mostly with matplotlib).
4. debugging/errorfinding. idk how to use the debugging tool, idwk how to use the debugging tool, AI fix (sometimes).

### Log
I start the work by doing the notbook save some of the single plotting and the collective plotting.\
This is due to me, at the time, being somewhat confused. As several of the columns did not appear to be suitable to plotting.\
I move onto starting the streamlit portion of the work, this included starting by familiarizing myself somewhat with sreamlit. I had\
joined the course late and missed the first lectures. Then I made my pages filled them with lorem ipsum and then had to make this confusing table thing.\
In the end it was only about turning the DF and compressing it into fewer cols and rows and then drawing the weird table. In this task again\
the question of what to tabulate or plot came up. I decided to temporarily make a smaller simpler selection of columns to make the architecture with.\
I then moved onto doing the plotting page where i first did select box -> months slider -> plot. But I wanted to have the months slider after the graph.\
This was done by storing in the sessionstate thus storing it through reruns. At this point i had done most of the task save for all the columns plotting\
and multiplotting on the notebook and streamlit. i then decided not all plots and did the five that made sense to plot.\
First i plotted everything as histograms, I later considered if I should switch to plotting lines against date

In [15]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from utils.fixit import signlog

In [16]:
reservoirs = pd.read_csv("../data/reservoirs.csv")
reservoirs = pd.DataFrame(reservoirs)
reservoirs.head()

,dato_Id,omrType,omrnr,iso_aar,iso_uke,fyllingsgrad,kapasitet_TWh,fylling_TWh,neste_Publiseringsdato,fyllingsgrad_forrige_uke,endring_fyllingsgrad
0,1995-09-03,EL,4,1995,35,0.944721,21.079208,19.913979,0001-01-01T00:00:00,0.936888,0.007833
1,2020-11-15,EL,1,2020,46,0.974361,6.003264,5.849344,2020-11-25T13:00:00,0.997406,-0.023046
2,2011-08-28,EL,4,2011,34,0.786499,21.079208,16.578772,0001-01-01T00:00:00,0.787193,-0.000694
3,1998-07-05,EL,3,1998,27,0.793969,8.920932,7.082947,0001-01-01T00:00:00,0.733812,0.060157
4,2011-03-27,EL,4,2011,12,0.300820,21.079208,6.341054,0001-01-01T00:00:00,0.311122,-0.010301


In [17]:
reservoirs = reservoirs.rename(columns={
    "dato_Id":"date (y-m-d)",
    "iso_aar":"year",
    "iso_uke":"week",
    "fyllingsgrad":"filled degree",
    "kapasitet_TWh":"capacity_Twh",
    "fylling_TWh":"fill_Twh",
    "neste_Publiseringsdato":"next_PublicizingDate",
    "fyllingsgrad_forrige_uke":"filled degree last week",
    "endring_fyllingsgrad":"change in filled degree"
})
reservoirs

,date (y-m-d),omrType,omrnr,year,week,filled degree,capacity_Twh,fill_Twh,next_PublicizingDate,filled degree last week,change in filled degree
0,1995-09-03,EL,4,1995,35,0.944721,21.079208,19.913979,0001-01-01T00:00:00,0.936888,0.007833
1,2020-11-15,EL,1,2020,46,0.974361,6.003264,5.849344,2020-11-25T13:00:00,0.997406,-0.023046
2,2011-08-28,EL,4,2011,34,0.786499,21.079208,16.578772,0001-01-01T00:00:00,0.787193,-0.000694
3,1998-07-05,EL,3,1998,27,0.793969,8.920932,7.082947,0001-01-01T00:00:00,0.733812,0.060157
4,2011-03-27,EL,4,2011,12,0.300820,21.079208,6.341054,0001-01-01T00:00:00,0.311122,-0.010301
...,...,...,...,...,...,...,...,...,...,...,...
14872,2026-07-19,VASS,3,2026,29,0.796849,28.155403,22.435612,2026-07-29T13:00:00,0.794957,0.001892
14873,2026-09-06,VASS,3,2026,36,0.826252,28.155403,23.263466,2026-09-16T13:00:00,0.827641,-0.001389
14874,2026-08-30,VASS,3,2026,35,0.827718,28.155403,23.304743,2026-09-09T13:00:00,0.829898,-0.002180
14875,2026-09-06,VASS,1,2026,36,0.570397,36.044464,20.559645,2026-09-16T13:00:00,0.567087,0.003310


In [18]:
#pandas plot backend
pd.options.plotting.backend = "plotly"

In [19]:
# removing columns not suitable for plotting
reservoirsCut = reservoirs[["filled degree", "capacity_Twh", "fill_Twh", "filled degree last week", "change in filled degree"]]
reservoirsCut

,filled degree,capacity_Twh,fill_Twh,filled degree last week,change in filled degree
0,0.944721,21.079208,19.913979,0.936888,0.007833
1,0.974361,6.003264,5.849344,0.997406,-0.023046
2,0.786499,21.079208,16.578772,0.787193,-0.000694
3,0.793969,8.920932,7.082947,0.733812,0.060157
4,0.300820,21.079208,6.341054,0.311122,-0.010301
...,...,...,...,...,...
14872,0.796849,28.155403,22.435612,0.794957,0.001892
14873,0.826252,28.155403,23.263466,0.827641,-0.001389
14874,0.827718,28.155403,23.304743,0.829898,-0.002180
14875,0.570397,36.044464,20.559645,0.567087,0.003310


In [20]:
# scaling columns
reservoirsLog = reservoirsCut.apply(signlog)

#plotting all columns
fig = go.Figure()

# fig1
dataFig1 = reservoirsLog["filled degree"]
kde = gaussian_kde(dataFig1)
x_range = np.linspace(dataFig1.min(), dataFig1.max(), 200)
density = kde(x_range)
density = density / density.max()
fig.add_trace(
    go.Scatter(x=x_range, y=density, fill="tozeroy", name="filled degree")
)

# fig2
dataFig2 = reservoirsLog["fill_Twh"]
kde = gaussian_kde(dataFig2)
x_range = np.linspace(dataFig2.min(), dataFig2.max(), 200)
density = kde(x_range)
density = density / density.max()
fig.add_trace(
    go.Scatter(x=x_range, y=density, fill="tozeroy", name="fill_Twh")
)

# fig3
dataFig3 = reservoirsLog["filled degree last week"]
kde = gaussian_kde(dataFig3)
x_range = np.linspace(dataFig3.min(), dataFig3.max(), 200)
density = kde(x_range)
density = density / density.max()
fig.add_trace(
    go.Scatter(x=x_range, y=density, fill="tozeroy", name="filled degree last week")
)

# fig4
dataFig4 = reservoirsLog["change in filled degree"]
kde = gaussian_kde(dataFig4)
x_range = np.linspace(dataFig4.min(), dataFig4.max(), 50)
density = kde(x_range)
density = density / density.max()
fig.add_trace(
    go.Scatter(x=x_range, y=density, fill="tozeroy", name="change in filled degree")
)

# fig5
dataFig5 = reservoirsLog["capacity_Twh"]
kde = gaussian_kde(dataFig5)
x_range = np.linspace(dataFig5.min(), dataFig5.max(), 200)
density = kde(x_range)
density = density / density.max()
fig.add_trace(
    go.Scatter(x=x_range, y=density, fill="tozeroy", name="capacity_Twh")
)

fig.show()
